# Lexical Processing: Tokens, Stemming, and Lexica

## MSA 8700 — Module 8: NLP and Text Processing

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what **tokenization** is and why it matters for NLP pipelines
2. Compare **stemming** and **lemmatization** — when to use each
3. Use **NLTK** for tokenization, stemming, and lemmatization
4. Navigate **WordNet** to find synonyms, antonyms, hypernyms, and more
5. Perform **Part-of-Speech (POS) tagging** and understand its applications
6. Understand how these classical techniques fit into **agentic AI workflows** alongside LLMs

---

# Part 1: Why Lexical Processing Still Matters

## Context: Before and Alongside LLMs

Before LLMs, most NLP systems relied on **bag-of-words** or **n-gram** features — representations that treat text as collections of individual words or short sequences. These required explicit preprocessing to normalize text.

Today, LLMs handle much of this internally. But classical lexical processing still appears in important components:

| Component | Why Lexical Processing Helps |
|-----------|----------------------------|
| **Search / Retrieval** | Stemming makes "deliveries" match "delivery" without needing an LLM |
| **Lightweight classification** | Keyword + stem matching can route 80% of queries without LLM cost |
| **Quality checks** | POS patterns can flag grammatically broken LLM outputs |
| **Synonym expansion** | WordNet synonyms broaden search queries deterministically |
| **Rule-based alerts** | Trigger on stemmed keywords (complain, delay, damage) with zero latency |

The theme is the same as the previous notebook: **use deterministic, fast methods for what they do well**, and reserve LLMs for reasoning and generation.

## Setup: Installing and Downloading NLTK Data

We use **NLTK** (Natural Language Toolkit), the most widely-used Python library for classical NLP. It provides tokenizers, stemmers, lemmatizers, POS taggers, and lexical databases like WordNet.

In [ ]:
# Install NLTK if not already available
# pip install nltk

import nltk

# Download required NLTK data (only needs to run once)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)

print("NLTK data downloaded successfully.")

---

# Part 2: Tokenization

## What Is Tokenization?

**Tokenization** is the process of splitting text into individual units called **tokens**. A token is typically a word, but can also be a punctuation mark, number, or other meaningful unit.

This is the very first step in almost every NLP pipeline — you cannot analyze words until you have identified what the words are.

### Why Not Just Split on Spaces?

You might think `text.split()` is sufficient. Let's see why it's not:

In [ ]:
text = "Dr. Smith's analysis isn't complete — she'll finish by 3:00 p.m."

# Naive approach: split on whitespace
naive_tokens = text.split()
print("Naive split():")
print(naive_tokens)
print(f"Count: {len(naive_tokens)} tokens\n")

In [ ]:
from nltk.tokenize import word_tokenize

# NLTK tokenizer: handles contractions, abbreviations, punctuation
nltk_tokens = word_tokenize(text)
print("NLTK word_tokenize():")
print(nltk_tokens)
print(f"Count: {len(nltk_tokens)} tokens")

### Key Differences

Notice what NLTK's tokenizer does that `split()` does not:

| Challenge | `split()` | `word_tokenize()` |
|-----------|-----------|-------------------|
| Contractions (`isn't`) | Keeps as one token | Splits into `is` + `n't` |
| Possessives (`Smith's`) | Keeps as one token | Splits into `Smith` + `'s` |
| Abbreviations (`Dr.`, `p.m.`) | Keeps period attached | Handles correctly |
| Punctuation (`—`) | Attached to adjacent word | Separated as its own token |

## Sentence Tokenization

Sometimes you need to split text into **sentences** rather than words. This is trickier than it sounds — not every period ends a sentence (think "Dr.", "U.S.", "3.14").

In [ ]:
from nltk.tokenize import sent_tokenize

paragraph = """Dr. Smith presented the Q1 results. Revenue grew by 12.5% vs. last year.
The U.S. market led growth. International markets showed mixed results."""

sentences = sent_tokenize(paragraph)
print(f"Found {len(sentences)} sentences:\n")
for i, sent in enumerate(sentences, 1):
    print(f"  {i}. {sent}")

Notice how the sentence tokenizer correctly handles "Dr.", "12.5%", and "U.S." as non-sentence-ending periods.

---

# Part 3: Stemming

## What Is Stemming?

**Stemming** chops off word endings to reduce inflected words to a common **stem**. The stem may not be a real word — it's a rough approximation designed for matching.

```
running   → run
deliveries → deliveri
complained → complain
happily   → happili
```

Stemming is **fast and rule-based** — it applies mechanical suffix-stripping rules without understanding language.

In [ ]:
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

text = "Customers complained that deliveries were delayed and the product was damaged."
tokens = word_tokenize(text)

stemmer = PorterStemmer()
stems = [stemmer.stem(t.lower()) for t in tokens]

# Display tokens and their stems side by side
print(f"{'Token':<15} {'Stem':<15}")
print("-" * 30)
for token, stem in zip(tokens, stems):
    marker = " ←" if token.lower() != stem else ""
    print(f"{token:<15} {stem:<15}{marker}")

The arrows mark words that were changed by stemming. Notice:
- `Customers` → `custom` (useful for matching)
- `complained` → `complain` (useful)
- `deliveries` → `deliveri` (not a real word, but matches other forms)
- `delayed` → `delay` (useful)
- `damaged` → `damag` (not a real word, but consistent)

## Why Stemming Is Useful: A Matching Example

Imagine a rule-based alert agent that needs to detect complaints about delays. Without stemming, you would need to enumerate all forms: "delay", "delayed", "delays", "delaying". With stemming, a single stem covers all of them.

In [ ]:
# Alert keywords (stemmed once)
alert_stems = {stemmer.stem(w) for w in ["complain", "delay", "damage", "cancel"]}
print(f"Alert stems: {alert_stems}\n")

# Incoming messages to check
messages = [
    "The customer is complaining about delayed deliveries again.",
    "Order shipped on time, everything looks good.",
    "Multiple cancellations reported due to damaged packaging.",
    "Quarterly revenue exceeded expectations.",
]

for msg in messages:
    msg_stems = {stemmer.stem(t.lower()) for t in word_tokenize(msg)}
    matches = alert_stems & msg_stems  # set intersection
    if matches:
        print(f"🚨 ALERT — matched {matches}")
        print(f"   Message: {msg}\n")
    else:
        print(f"   OK: {msg}\n")

## Comparing Stemmers

NLTK provides several stemmers. The **Porter Stemmer** is the most common, but others exist for different trade-offs:

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

porter = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()

test_words = ["running", "deliveries", "happily", "organization", "studies", "generously", "fairly"]

print(f"{'Word':<16} {'Porter':<14} {'Snowball':<14} {'Lancaster':<14}")
print("-" * 58)
for word in test_words:
    print(f"{word:<16} {porter.stem(word):<14} {snowball.stem(word):<14} {lancaster.stem(word):<14}")

| Stemmer | Aggressiveness | Notes |
|---------|---------------|-------|
| **Porter** | Moderate | Most widely used; good balance of precision and recall |
| **Snowball** | Moderate | Improved version of Porter; supports multiple languages |
| **Lancaster** | Aggressive | Strips more aggressively; sometimes over-stems |

---

# Part 4: Lemmatization

## Stemming vs. Lemmatization

**Lemmatization** is the smarter cousin of stemming. Instead of mechanically chopping suffixes, it uses a **dictionary and morphological analysis** to reduce words to their actual base form (the **lemma**).

| | Stemming | Lemmatization |
|--|---------|---------------|
| **Method** | Rule-based suffix stripping | Dictionary lookup + morphology |
| **Output** | May not be a real word (`deliveri`) | Always a valid word (`delivery`) |
| **Speed** | Very fast | Slightly slower |
| **Needs POS?** | No | Yes (for best results) |
| **Use case** | Search indexing, quick matching | Text analysis, feature engineering |

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

# Compare stemming vs. lemmatization
words = ["running", "deliveries", "complained", "happily", "better", "geese", "studies", "wolves"]

print(f"{'Word':<15} {'Stem':<15} {'Lemma (noun)':<15} {'Lemma (verb)':<15}")
print("-" * 60)
for word in words:
    stem = stemmer.stem(word)
    lemma_n = lemmatizer.lemmatize(word, pos='n')  # as noun
    lemma_v = lemmatizer.lemmatize(word, pos='v')  # as verb
    print(f"{word:<15} {stem:<15} {lemma_n:<15} {lemma_v:<15}")

### Key Observations

- **Lemmatization produces real words**: `deliveries` → `delivery` (not `deliveri`)
- **POS matters**: `running` lemmatized as a **noun** stays `running` (it's a valid noun: "the running of the bulls"), but as a **verb** it becomes `run`
- **Irregular forms**: Lemmatization handles `geese` → `goose` and `better` → `good` (when tagged correctly), which stemming cannot
- **Stemming is still useful** when you don't need real words — just consistent matching

## Putting It Together: Tokenize → POS Tag → Lemmatize

For the best lemmatization results, the standard pipeline is:
1. **Tokenize** the text
2. **POS tag** each token (so the lemmatizer knows if it's a noun, verb, adjective, etc.)
3. **Lemmatize** using the correct POS

In [ ]:
from nltk import pos_tag
from nltk.corpus import wordnet as wn

def get_wordnet_pos(treebank_tag):
    """Convert NLTK POS tags to WordNet POS tags."""
    if treebank_tag.startswith('J'):
        return wn.ADJ
    elif treebank_tag.startswith('V'):
        return wn.VERB
    elif treebank_tag.startswith('N'):
        return wn.NOUN
    elif treebank_tag.startswith('R'):
        return wn.ADV
    else:
        return wn.NOUN  # default to noun


text = "The customers were complaining that their deliveries arrived damaged and delayed."

# Step 1: Tokenize
tokens = word_tokenize(text)

# Step 2: POS tag
tagged = pos_tag(tokens)

# Step 3: Lemmatize with correct POS
lemmatizer = WordNetLemmatizer()
results = []
for word, tag in tagged:
    wn_pos = get_wordnet_pos(tag)
    lemma = lemmatizer.lemmatize(word.lower(), pos=wn_pos)
    results.append((word, tag, lemma))

print(f"{'Token':<15} {'POS Tag':<10} {'Lemma':<15}")
print("-" * 40)
for word, tag, lemma in results:
    marker = " ←" if word.lower() != lemma else ""
    print(f"{word:<15} {tag:<10} {lemma:<15}{marker}")

Now every word is reduced to its proper dictionary form: `were` → `be`, `complaining` → `complain`, `deliveries` → `delivery`, `arrived` → `arrive`, etc.

---

# Part 5: Part-of-Speech (POS) Tagging

## What Is POS Tagging?

**Part-of-Speech tagging** assigns a grammatical category to each word in a sentence: noun, verb, adjective, adverb, etc.

This is useful on its own — not just as a preprocessing step for lemmatization.

In [ ]:
from nltk import pos_tag
from nltk.tokenize import word_tokenize

sentence = "The quick brown fox jumps over the lazy dog."
tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)

print("POS Tags:")
for word, tag in tagged:
    print(f"  {word:<10} {tag}")

## Common POS Tags (Penn Treebank)

| Tag | Meaning | Examples |
|-----|---------|----------|
| `NN` | Noun, singular | dog, city, music |
| `NNS` | Noun, plural | dogs, cities |
| `NNP` | Proper noun | John, London |
| `VB` | Verb, base form | run, eat, be |
| `VBD` | Verb, past tense | ran, ate, was |
| `VBG` | Verb, gerund | running, eating |
| `VBZ` | Verb, 3rd person singular | runs, eats |
| `JJ` | Adjective | quick, brown, lazy |
| `RB` | Adverb | quickly, very, well |
| `DT` | Determiner | the, a, an |
| `IN` | Preposition | in, on, over |
| `PRP` | Personal pronoun | I, he, she |

## Practical Application: Extracting Descriptors

POS tags let you extract specific grammatical patterns. For example, extracting **adjective + noun** pairs from product reviews gives you product descriptors.

In [ ]:
reviews = [
    "The ergonomic keyboard has excellent build quality and responsive keys.",
    "Compact design with a beautiful display, but the battery life is short.",
    "Affordable price for a durable product with fast shipping.",
]

print("Adjective + Noun descriptors extracted from reviews:\n")

for review in reviews:
    tokens = word_tokenize(review)
    tagged = pos_tag(tokens)
    
    # Find [JJ] [NN/NNS] patterns (adjective followed by noun)
    descriptors = []
    for i in range(len(tagged) - 1):
        word1, tag1 = tagged[i]
        word2, tag2 = tagged[i + 1]
        if tag1 == 'JJ' and tag2 in ('NN', 'NNS'):
            descriptors.append(f"{word1} {word2}")
    
    if descriptors:
        print(f"  Review: \"{review[:60]}...\"")
        print(f"  Descriptors: {descriptors}\n")

> **Agentic use case**: A product analytics agent can extract adjective-noun pairs from thousands of reviews to build a **feature-sentiment map** without any LLM calls — then send the summarized descriptors to an LLM for higher-level insights.

---

# Part 6: WordNet — A Lexical Database

## What Is WordNet?

**WordNet** is a large, human-curated lexical database of English. It groups words into sets of synonyms called **synsets** and records relationships between them:

| Relationship | Meaning | Example |
|-------------|---------|--------|
| **Synonyms** | Words with similar meaning | buy, purchase, acquire |
| **Antonyms** | Words with opposite meaning | buy ↔ sell |
| **Hypernyms** | More general category ("is-a") | dog → canine → animal |
| **Hyponyms** | More specific category | animal → dog, cat, horse |
| **Meronyms** | Part-of relationship | car → wheel, engine, door |

## Example 1: Finding Synonyms

A core WordNet use case: expanding a single keyword into a set of synonyms for broader matching.

In [ ]:
from nltk.corpus import wordnet as wn

word = "buy"
synonyms = set()

# wn.synsets() returns all synsets (meaning groups) for a word
for syn in wn.synsets(word, pos=wn.VERB):
    # Each synset has one or more lemmas (word forms)
    for lemma in syn.lemmas():
        synonyms.add(lemma.name())

print(f'Synonyms of "{word}" (verbs):')
print(f"  {synonyms}")

### Breaking Down the Code

```python
wn.synsets(word, pos=wn.VERB)
```
Returns a list of **synsets** — groups of words that share a meaning. Each synset represents one sense of the word. For example, "buy" as in "purchase something" is a different synset from "buy" as in "accept as true" ("I don't buy that argument").

```python
syn.lemmas()
```
Returns the **lemmas** (word forms) in that synset. Each lemma is one synonym.

## Example 2: Exploring Synsets in Detail

Let's look more closely at what a synset contains.

In [ ]:
word = "bank"

print(f'All synsets for "{word}":\n')
for syn in wn.synsets(word):
    lemma_names = [l.name() for l in syn.lemmas()]
    print(f"  {syn.name():<25} ({syn.pos()})  {syn.definition()}")
    print(f"  {'':25} Lemmas: {lemma_names}\n")

This shows that "bank" is **polysemous** — it has multiple distinct meanings (financial institution, river bank, etc.). Each meaning is a separate synset with its own definition and set of synonyms.

This is why the `pos` parameter matters: filtering by part of speech narrows down to relevant meanings.

## Example 3: Antonyms

Antonyms are stored at the **lemma** level (not the synset level), because antonymy is a relationship between specific words.

In [ ]:
# Find antonyms for several words
test_words = ["good", "buy", "increase", "happy", "fast"]

print("Antonyms:\n")
for word in test_words:
    antonyms = set()
    for syn in wn.synsets(word):
        for lemma in syn.lemmas():
            if lemma.name() == word:  # only look at lemmas matching our word
                for ant in lemma.antonyms():
                    antonyms.add(ant.name())
    if antonyms:
        print(f"  {word:<12} ↔  {antonyms}")
    else:
        print(f"  {word:<12} ↔  (no direct antonym found)")

## Example 4: Hypernyms and Hyponyms (Is-A Hierarchy)

WordNet organizes nouns and verbs into a **taxonomy** — a tree of increasingly general categories.

In [ ]:
# Start with "dog" and walk up the hypernym tree
dog = wn.synset('dog.n.01')

print(f"Starting synset: {dog.name()} — {dog.definition()}\n")

print("Hypernym chain (more general):")
current = dog
depth = 0
while current.hypernyms():
    current = current.hypernyms()[0]
    depth += 1
    print(f"  {'  ' * depth}↑ {current.name():<30} {current.definition()}")

print(f"\nDirect hyponyms of 'dog' (more specific — first 8):")
for hypo in dog.hyponyms()[:8]:
    print(f"  ↓ {hypo.name():<30} {hypo.definition()[:60]}")

## Example 5: Building a Synonym Lexicon for an Agent

Here's the practical agentic use case from the lecture notes: building a **lexicon of "buy-like" verbs** that an agent can use for deterministic keyword matching.

In [ ]:
def build_synonym_lexicon(seed_words, pos=wn.VERB):
    """Build a set of synonyms from seed words using WordNet."""
    lexicon = set()
    for word in seed_words:
        for syn in wn.synsets(word, pos=pos):
            for lemma in syn.lemmas():
                # Replace underscores with spaces for multi-word expressions
                lexicon.add(lemma.name().replace('_', ' '))
    return lexicon

# Build a "purchase intent" lexicon
purchase_lexicon = build_synonym_lexicon(["buy", "purchase", "order"])
print("Purchase intent lexicon:")
print(f"  {sorted(purchase_lexicon)}\n")

# Build a "complaint" lexicon
complaint_lexicon = build_synonym_lexicon(["complain", "protest", "object"])
print("Complaint lexicon:")
print(f"  {sorted(complaint_lexicon)}")

In [ ]:
# Use the lexicon to classify customer messages
messages = [
    "I'd like to purchase two units of the Pro model.",
    "We want to acquire the enterprise license.",
    "Can I get a refund for order #12345?",
    "The weather is nice today.",
    "Please order 50 widgets for our warehouse.",
]

stemmer = PorterStemmer()
purchase_stems = {stemmer.stem(w) for w in purchase_lexicon}

print("Classifying messages for purchase intent:\n")
for msg in messages:
    msg_stems = {stemmer.stem(t.lower()) for t in word_tokenize(msg)}
    matches = purchase_stems & msg_stems
    has_intent = len(matches) > 0
    label = "PURCHASE INTENT" if has_intent else "no intent"
    print(f"  [{label:>16}]  {msg}")
    if has_intent:
        print(f"  {'':>18}  matched stems: {matches}")

> **Agentic use case**: This agent monitors customer feedback for purchase intent. It uses a **WordNet-expanded lexicon** combined with **stemming** for matching. The LLM is only called for borderline or ambiguous cases — saving cost and latency for the clear-cut majority.

---

# Part 7: Stop Words

## What Are Stop Words?

**Stop words** are extremely common words ("the", "is", "at", "in", "a") that carry little meaning on their own. In many NLP tasks — especially bag-of-words models and keyword extraction — removing them reduces noise and improves signal.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

print(f"NLTK English stop words ({len(stop_words)} total):")
print(f"  Sample: {sorted(list(stop_words))[:20]}...\n")

# Filter stop words from a sentence
text = "The customers were very unhappy with the slow delivery of their orders."
tokens = word_tokenize(text.lower())

filtered = [t for t in tokens if t not in stop_words and t.isalpha()]

print(f"Original tokens:  {[t for t in tokens if t.isalpha()]}")
print(f"After filtering:  {filtered}")

Notice how the filtered version retains the **content words** — the words that carry actual meaning: `customers`, `unhappy`, `slow`, `delivery`, `orders`.

---

# Part 8: Putting It All Together

Let's build a complete **lexical preprocessing pipeline** that combines everything we've learned.

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet as wn
from nltk import pos_tag


def get_wordnet_pos(treebank_tag):
    """Convert NLTK POS tags to WordNet POS tags."""
    if treebank_tag.startswith('J'):
        return wn.ADJ
    elif treebank_tag.startswith('V'):
        return wn.VERB
    elif treebank_tag.startswith('N'):
        return wn.NOUN
    elif treebank_tag.startswith('R'):
        return wn.ADV
    else:
        return wn.NOUN


def preprocess_text(text):
    """Full lexical preprocessing pipeline."""
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    # Step 1: Sentence tokenization
    sentences = sent_tokenize(text)
    
    all_results = []
    for sent in sentences:
        # Step 2: Word tokenization
        tokens = word_tokenize(sent)
        
        # Step 3: POS tagging
        tagged = pos_tag(tokens)
        
        # Step 4: Lemmatization + stop word removal
        processed = []
        for word, tag in tagged:
            word_lower = word.lower()
            if word_lower in stop_words or not word.isalpha():
                continue
            wn_pos = get_wordnet_pos(tag)
            lemma = lemmatizer.lemmatize(word_lower, pos=wn_pos)
            processed.append(lemma)
        
        all_results.append({
            'sentence': sent,
            'lemmas': processed
        })
    
    return all_results


# Process a customer review
review = """The customers complained that their deliveries were consistently delayed.
Several products arrived damaged and poorly packaged. Management has been
unresponsive to repeated complaints about the declining service quality."""

results = preprocess_text(review)

print("Preprocessed review:\n")
for r in results:
    print(f"  Original: {r['sentence']}")
    print(f"  Lemmas:   {r['lemmas']}\n")

---

# Practice Exercises

Try these exercises to reinforce what you've learned.

## Exercise 1: Stemming vs. Lemmatization

For each word below, compute both the Porter stem and the WordNet lemma (using the correct POS). Which approach produces more readable output?

In [ ]:
words_with_pos = [
    ("running", wn.VERB),
    ("better", wn.ADJ),
    ("universities", wn.NOUN),
    ("organizing", wn.VERB),
    ("mice", wn.NOUN),
    ("happily", wn.ADV),
]

# TODO: For each (word, pos) pair, print the Porter stem and WordNet lemma
# Your code here

## Exercise 2: Build a Custom Alert Lexicon

Use WordNet to build a lexicon of "negative service" words starting from the seeds: `["delay", "damage", "cancel", "fail"]`. Then test it against the sample messages.

In [ ]:
seed_words = ["delay", "damage", "cancel", "fail"]

messages = [
    "The shipment was postponed again due to supplier issues.",
    "Everything arrived on time and in perfect condition.",
    "Our system crashed and we lost the backup data.",
    "The project was called off after budget cuts.",
]

# TODO:
# 1. Build a synonym lexicon from the seed words using WordNet
# 2. Check each message against the lexicon (using stemming for matching)
# 3. Print which messages trigger an alert and which matched stems

# Your code here

## Exercise 3: Extract Noun Phrases

Use POS tagging to extract all **noun phrases** (sequences of adjectives and nouns) from the product description below.

In [ ]:
description = """The lightweight wireless headphones feature premium sound quality
with active noise cancellation. The ergonomic design provides comfortable
all-day wear and the long battery life ensures uninterrupted listening."""

# TODO:
# 1. Tokenize and POS tag the text
# 2. Extract sequences matching (JJ)* (NN|NNS)+ patterns
#    (zero or more adjectives followed by one or more nouns)
# 3. Print the extracted noun phrases

# Your code here

---

## Summary

| Technique | What It Does | Key Function / Class |
|-----------|-------------|---------------------|
| **Tokenization** | Splits text into words or sentences | `word_tokenize()`, `sent_tokenize()` |
| **Stemming** | Chops suffixes to get rough word stems (fast, may not be real words) | `PorterStemmer`, `SnowballStemmer` |
| **Lemmatization** | Reduces words to dictionary base form (slower, always valid words) | `WordNetLemmatizer` |
| **POS Tagging** | Labels words with grammatical categories (noun, verb, adjective...) | `pos_tag()` |
| **WordNet** | Lexical database for synonyms, antonyms, hypernyms, and more | `wordnet.synsets()`, `.lemmas()` |
| **Stop Words** | Removes common low-information words | `stopwords.words('english')` |

### Key Takeaways

1. **Tokenization** is the essential first step — `split()` is not enough for real text
2. **Stemming** is fast and great for matching; **lemmatization** is better when you need real words
3. **POS tagging** enables grammatical pattern extraction (adjective-noun descriptors, verb phrases)
4. **WordNet** lets you build synonym lexicons deterministically — expanding keyword coverage without LLM calls
5. These classical techniques are **complementary to LLMs**: use them for fast, deterministic, low-cost preprocessing, and reserve LLMs for reasoning and generation